<a href="https://colab.research.google.com/github/Auta01/Pytorch/blob/main/Text%20classification%20project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers --quiet
!pip install opendatasets --quiet

import opendatasets as od
od.download('https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: Autao1
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection


100%|██████████| 3.30M/3.30M [00:00<00:00, 178MB/s]

In [3]:
import torch
from torch import nn
from torch.optim import Adam
from transformers import  AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
data_df = pd.read_json('/content/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json', lines = True)
data_df.dropna(inplace = True)
data_df.drop_duplicates(inplace = True)
data_df.drop('article_link', inplace = True, axis=1)
print(data_df.shape)
data_df.head()

(26708, 2)


,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


In [20]:
# First split: 70% train, 30% for a temporary test/validation pool
x_train, x_temp_test, y_train, y_temp_test = train_test_split(
    np.array(data_df['headline']),
    np.array(data_df['is_sarcastic']),
    test_size=0.3,
    random_state=42 # Added for reproducibility
)

# Second split: From the 30% pool, split it 50/50 for validation and final test sets
x_val, x_test, y_val, y_test = train_test_split(
    x_temp_test, # Use the temporary headlines for the next split
    y_temp_test, # Use the temporary labels for the next split
    test_size=0.5, # 50% of the 30% for x_test (15% overall) and 50% for x_val (15% overall)
    random_state=42 # Added for reproducibility
)

print('Training size:',x_train.shape[0], 'rows which is ', round((x_train.shape[0]/data_df.shape[0])*100, 2), '%')
print('Validation size:',x_val.shape[0], 'rows which is ', round((x_val.shape[0]/data_df.shape[0])*100, 2), '%')
print('Testing size:', x_test.shape[0], 'rows which is ', round((x_test.shape[0]/data_df.shape[0])*100, 2), '%')

Training size: 18695 rows which is  70.0 %
Validation size: 4006 rows which is  15.0 %
Testing size: 4007 rows which is  15.0 %


In [12]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
class datasets(Dataset): # Corrected base class to Dataset
    def __init__(self, x_data, y_data): # Renamed parameters for clarity
        # Tokenize inputs (x_data)
        tokenized_inputs = tokenizer( # Using the global tokenizer object
            x_data.tolist() if isinstance(x_data, np.ndarray) else x_data, # Ensure input is a list of strings
            max_length=100,
            truncation=True,
            padding='max_length', # Use 'max_length' for padding
            return_tensors='pt' # Corrected syntax: 'pt' is a string argument
        )
        # Store tokenized inputs as self.x, moving to device
        # The original code had self.x as the tokenized data, so keeping a similar structure.
        self.x_input_ids = tokenized_inputs['input_ids'].to(device)
        self.x_attention_mask = tokenized_inputs['attention_mask'].to(device)

        # Store labels as self.y, moving to device
        self.y = torch.tensor(y_data, dtype=torch.long).to(device) # Using torch.tensor and moving to device

    def __len__(self): # Added colon
      return len(self.y) # Length should be based on labels or input_ids

    def __getitem__(self, index): # Added colon
      # Return input_ids, attention_mask, and label for the given index
      return {
          'input_ids': self.x_input_ids[index],
          'attention_mask': self.x_attention_mask[index],
          'labels': self.y[index]
      }


Training_data = datasets(x_train, y_train)
validation_data = datasets(x_val, y_val)
testing_data = datasets(x_test, y_test)

In [17]:
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4


In [21]:
train_dataloader = DataLoader(Training_data, batch_size = BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size = BATCH_SIZE, shuffle=True)
testing_dataloader = DataLoader(testing_data, batch_size = BATCH_SIZE, shuffle=True)

In [30]:
class model(nn.Module): # Corrected to nn.Module
  def __init__(self): # Added space and colon
    super(model,self).__init__()
    self.bert = bert_model
    self.dropout = nn.Dropout(0.25)
    self.linear1 = nn.Linear(768,384) # Corrected casing for consistency
    self.linear2 = nn.Linear(384,1) # Corrected casing for consistency
    self.sigmoid = nn.Sigmoid()

  def forward(self, input_id, attention_mask):
    # Correct way to call the BERT model and get the pooled output
    # `return_dict=False` makes it return a tuple, where the second element is the pooled output
    outputs = self.bert(input_ids=input_id, attention_mask=attention_mask, return_dict=False)
    pooled_output = outputs[1] # Get the pooled output for classification

    output = self.linear1(pooled_output) # Use correct casing for linear1 and the pooled_output
    output = self.dropout(output)
    output = self.linear2(output) # Use correct casing for linear2
    output = self.sigmoid(output)
    return output

In [34]:
for param in bert_model.parameters():
  param.requires_grad = False
model = model()

In [35]:
model


model(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True

In [ ]:
loss_fn = BCEloss
OPTIMIZER = optimizer.Adam.